# 03 검색 결과를 비교하고 구현으로 넘어가기

**필수(15분)**: 질문 표현이나 검색 조건을 하나만 바꿔 반환된 조각 ID·원문·출처가 어떻게 달라지는지 봅니다. 결과가 같으면 같은 그대로 보여 줍니다. 필수 셀을 마치면 바로 **문서 RAG 바이브코딩**으로 넘어갑니다.

**선택**: BM25·벡터·MMR 비교, 메타데이터 필터, 질문 임베딩 3D는 아래 ‘선택’ 셀입니다. 수업 시간에는 건너뜁니다.

- 문서: 삼일PwC 『반도체 산업 Business Guidebook』(2025.12) **책자 6쪽 = PDF 8번째 페이지** ‘2. 반도체 Value Chain Overview (8대 공정)’ · [발행처 원문(8번째 페이지)](https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf#page=8)
- 원본 PDF는 함께 배포하지 않습니다. 노트북 옆 `data/samilpwc_semicon-guidebook.pdf`가 없으면 발행처 주소에서 내려받습니다. 내려받을 수 없는 환경이면 위 링크에서 직접 받아 그 위치에 두세요.
- 앞 챕터를 실행하지 않아도 됩니다. 새 커널에서 첫 셀부터 차례로 실행합니다.
- 이 노트북은 임베딩 모델(text-embedding-3-small)만 씁니다.

In [ ]:
# 현재 Python 3.12 커널에 실습 패키지를 설치합니다(처음 한 번).
# 설치 후 커널 재시작 안내가 나오면 재시작하고 다음 셀부터 실행하세요.
%pip install langchain-core==1.8.4 langchain-openai==1.8.5 langchain-chroma==1.2.0 chromadb==1.5.9 openai==3.59.0 pypdf==7.1.1 numpy==2.5.5 pandas==3.1.1 scikit-learn==1.9.1 plotly==7.2.0

In [ ]:
# ▶ 설정 셀 — 따옴표 안의 값만 바꾸세요. 모델명은 확정값이라 바꾸지 않습니다.
# 키가 든 이 셀은 화면 공유·캡처하거나 AI 도우미에 붙여넣지 않습니다. 파일을 공유할 때는 자리표시 문자열로 되돌립니다.
API_KEY = "여기에_수업용_API_키_입력"
EMBEDDING_BASE_URL = "여기에_임베딩용_BASE_URL_입력"    # 안내받은 임베딩 모델 주소(https://..., 채팅 주소와 다름)
EMBEDDING_MODEL_NAME = "text-embedding-3-small"        # 문서·질문 임베딩 모델(확정값)

In [ ]:
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
NEEDED = ["API_KEY", "EMBEDDING_BASE_URL"]
missing = [name for name in NEEDED if globals()[name].startswith("여기에") or not globals()[name].strip()]
if missing:   # 임베딩 주소가 없을 때 채팅 주소로 대신하지 않는다
    raise ValueError(", ".join(missing) + "에 안내받은 값을 설정 셀의 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME, api_key=API_KEY, base_url=EMBEDDING_BASE_URL,
                              check_embedding_ctx_length=False)
print("임베딩 모델:", EMBEDDING_MODEL_NAME, "· 벡터 길이:", len(embeddings.embed_query("시험")))

## 필수 1. 문서 준비(01과 같은 방법)

In [ ]:
import os, urllib.request
from pathlib import Path

PDF_URL = "https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf"
PDF_FILE = Path("data/samilpwc_semicon-guidebook.pdf")       # 원본은 함께 배포하지 않음 → 없으면 내려받음
SOURCE = "삼일PwC 『반도체 산업 Business Guidebook』(2025.12)"
PAGE_INDEX = 7                       # 파이썬은 0부터 셈 → PDF 뷰어의 8번째 페이지
PDF_PAGE, BOOK_PAGE = PAGE_INDEX + 1, 6   # 책자에 인쇄된 쪽 번호는 6
VIEW_URL = f"{PDF_URL}#page={PDF_PAGE}"   # #page=8은 뷰어 이동용. 코드에서는 PAGE_INDEX로 페이지를 고른다

def ensure_pdf():
    if not PDF_FILE.is_file():
        PDF_FILE.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(PDF_URL, timeout=60) as r:     # 기본 요청으로 받는다(브라우저 흉내 헤더는 거절됨)
                PDF_FILE.write_bytes(r.read())
            print("발행처에서 내려받음:", PDF_URL)
        except Exception as e:
            raise RuntimeError(f"PDF를 받지 못했습니다({type(e).__name__}). 브라우저로 {PDF_URL} 을 내려받아 "
                               f"이 노트북 옆 {PDF_FILE} 에 두고 다시 실행하세요.")
    return PDF_FILE

from pypdf import PdfReader

def page_runs(page):
    """페이지의 글자 묶음마다 (글자 크기, 세로 위치, 글자)를 모은다."""
    runs = []
    page.extract_text(visitor_text=lambda t, cm, tm, fd, fs: runs.append((round(fs * (tm[0] or 1), 1), tm[5], t)))
    return runs

def read_body(page):
    """제목(20pt)과 본문(10pt, 공정 그림 아래쪽)만 남기고 그림 속 짧은 이름표는 뺀다."""
    runs = page_runs(page)
    title = "".join(t for size, y, t in runs if size >= 19).replace("\n", "").strip()
    body = "".join(t for size, y, t in runs if 9.5 <= size <= 10.5 and y < 360).replace("\n", "").strip()
    return title, body

from langchain_core.documents import Document

def split_text(text, size, overlap):
    """size 글자마다 자르고, 다음 조각은 앞 조각의 끝 overlap 글자부터 시작한다 → (시작, 끝) 목록"""
    assert 0 <= overlap < size, "겹침은 0 이상, 조각 크기보다 작아야 합니다"
    spans, start = [], 0
    while True:
        end = min(start + size, len(text))
        spans.append((start, end))
        if end >= len(text):
            return spans
        start += size - overlap

def make_chunks(title, body, size, overlap):
    return [Document(page_content=body[s:e], metadata={
                "chunk_id": f"pwc-p{PDF_PAGE}-c{i:02d}", "source": SOURCE, "title": title, "file": PDF_FILE.name,
                "book_page": BOOK_PAGE, "pdf_page": PDF_PAGE, "url": VIEW_URL, "start": s, "end": e,
                "chunk_size": size, "overlap": overlap})
            for i, (s, e) in enumerate(split_text(body, size, overlap), start=1)]

import hashlib
from langchain_chroma import Chroma

def open_index(chunks):
    """설정(조각 크기·겹침)·본문·임베딩 모델이 같으면 같은 컬렉션 이름 → 다시 실행해도 중복 저장하지 않는다."""
    m = chunks[0].metadata
    key = EMBEDDING_MODEL_NAME + "".join(c.page_content for c in chunks)
    name = f"pwc-p{PDF_PAGE}-{m['chunk_size']}-{m['overlap']}-{hashlib.sha256(key.encode('utf-8')).hexdigest()[:8]}"
    store = Chroma(collection_name=name, embedding_function=embeddings, persist_directory="chroma_db",
                   collection_metadata={"hnsw:space": "cosine"})            # 거리 = 코사인 거리(작을수록 가까움)
    ids = [c.metadata["chunk_id"] for c in chunks]
    have = set(store.get(ids=ids)["ids"])
    new = [c for c in chunks if c.metadata["chunk_id"] not in have]
    if new:
        store.add_documents(new, ids=[c.metadata["chunk_id"] for c in new])
    print(f"컬렉션 {name} · 새로 추가 {len(new)}개 · 저장된 조각 {len(store.get()['ids'])}개")
    return store

def search(store, question, k=3, **kw):
    found = store.similarity_search_with_score(question, k=k, **kw)
    return [{"rank": i, "distance": round(float(d), 4), "text": doc.page_content, **doc.metadata}
            for i, (doc, d) in enumerate(found, start=1)]

def show_hits(hits):
    for h in hits:
        print(f"{h['rank']}위 [{h['chunk_id']}] 거리 {h['distance']} · {h['source']} 책자 {h['book_page']}쪽 = PDF {h['pdf_page']}번째 페이지"
              f" · 본문 {h['start']}~{h['end']}자")
        print("    " + h["text"])

reader = PdfReader(str(ensure_pdf()))
TITLE, BODY = read_body(reader.pages[PAGE_INDEX])
CHUNKS = make_chunks(TITLE, BODY, 200, 60)
store = open_index(CHUNKS)
print(f"문서: {SOURCE} · 책자 {BOOK_PAGE}쪽(PDF {PDF_PAGE}번째 페이지) · ‘{TITLE}’ · 본문 {len(BODY)}자 · 조각 {len(CHUNKS)}개")
ETCH = "식각 공정에서는 포토공정에서 입혀진 감광액을 제외한 부분을 깎아내게 되며, 이후 증착 및 이온주입공정을 통해 전기적 특성을 부여하게 된다"

import pandas as pd

def table(hits, base=None):
    """검색 결과 표. base(기준 결과)가 있으면 새로 들어온 조각·빠진 조각·순위 이동을 표시한다."""
    base_rank = {h["chunk_id"]: h["rank"] for h in (base or [])}
    rows = [{"순위": h["rank"], "조각 ID": h["chunk_id"], "거리": h["distance"], "출처": f"책자 {h['book_page']}쪽·PDF {h['pdf_page']}쪽",
             "식각 문장 온전히": "✔" if ETCH in h["text"] else "", "기준과 비교": "" if base is None else
             ("새로 포함" if h["chunk_id"] not in base_rank else ("같은 순위" if base_rank[h["chunk_id"]] == h["rank"]
              else f"{base_rank[h['chunk_id']]}위 → {h['rank']}위")), "원문 앞부분": h["text"][:50]} for h in hits]
    if base is not None:
        now = {h["chunk_id"] for h in hits}
        rows += [{"순위": "—", "조각 ID": cid, "기준과 비교": f"빠짐(기준 {r}위)"} for cid, r in base_rank.items() if cid not in now]
    return pd.DataFrame(rows).astype(object).fillna("")

## 필수 2. 질문 표현만 바꾸기

같은 색인·같은 k=3에서 질문 문장만 바꿉니다. 첫 질문의 결과를 기준으로 새로 포함·빠짐·순위 이동을 표시합니다.

In [ ]:
base_q = "웨이퍼에서 원하는 무늬만 남기고 필요 없는 부분을 깎는 공정은 뭐야?"
base_hits = search(store, base_q, k=3)
print("기준 질문:", base_q)
table(base_hits)

In [ ]:
for q in ["식각 공정이란?", "감광액이 덮이지 않은 부분을 없애는 단계는?"]:
    print("바꾼 질문:", q)
    display(table(search(store, q, k=3), base=base_hits))

## 필수 3. 검색 조건 하나만 바꾸기 — 조각 겹침 60자 → 0자

질문은 그대로 두고 조각 겹침만 0으로 바꿔 **다른 컬렉션**에 색인합니다. 조각 경계가 달라지면 같은 조각 ID라도 내용이 다릅니다. ‘식각 문장 온전히’ 열을 비교합니다.

In [ ]:
store0 = open_index(make_chunks(TITLE, BODY, 200, 0))
hits0 = search(store0, base_q, k=3)
print("겹침 60자(기준)"); display(table(base_hits))
print("겹침 0자"); display(table(hits0))

## 필수 4. 영상·그림 — 질문 표현을 바꿔도 같은 근거로 돌아오는가

영상: [MIT OCW 6.774 Lecture 20 Etching – Introduction](https://www.youtube.com/watch?v=4q5k1hLfyrM) (자막: 같은 영상을 임베드한 [MIT 공식 WebVTT](https://ocw.mit.edu/courses/6-774-physics-of-microfabrication-front-end-processing-fall-2004/1qv-RkuGGz7mFmOaR3Kq1JxAKb6ThWhhd_transcript.webvtt), 영상 설명의 이용 조건 CC BY-NC-SA — 기업 교육에서 자막을 재사용할 때는 허용 범위를 확인) · 그림: [삼성반도체 「반도체 8대 공정 5탄 ‘식각 공정’」](https://semiconductor.samsung.com/kr/support/tools-resources/fabrication-process/eight-essential-semiconductor-fabrication-processes-part-5-etching-a-circuit-pattern/) 본문 그림의 캡션과 대체 텍스트. 임베딩에는 **글자(자막·캡션·대체 텍스트)만** 넣습니다. 이미지 자체를 분석하거나 검색하지 않습니다.

In [ ]:
import html, re, urllib.request
from langchain_core.documents import Document

VIDEO_ID = "4q5k1hLfyrM"
VIDEO_TITLE = "MIT OCW 6.774 Lecture 20: Etching – Introduction (Judy Hoyt)"
VIDEO_PAGE = f"https://www.youtube.com/watch?v={VIDEO_ID}"
MIT_VTT = ("https://ocw.mit.edu/courses/6-774-physics-of-microfabrication-front-end-processing-fall-2004/"
           "1qv-RkuGGz7mFmOaR3Kq1JxAKb6ThWhhd_transcript.webvtt")          # 같은 영상을 임베드한 MIT 공식 페이지의 영어 자막
ARTICLE_URL = ("https://semiconductor.samsung.com/kr/support/tools-resources/fabrication-process/"
               "eight-essential-semiconductor-fabrication-processes-part-5-etching-a-circuit-pattern/")
ARTICLE_TITLE = "삼성반도체 「반도체 8대 공정 5탄, 반도체 회로패턴의 완성 ‘식각 공정’」"

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read().decode("utf-8", errors="replace")

def seconds(t):                       # "00:02:00.370" → 120.37
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

def caption_text():
    """유튜브 자막을 먼저 요청하고, 비어 있거나 실패하면 MIT 공식 WebVTT를 쓴다. (자막 원문, 자막 출처)"""
    try:
        body = fetch(f"https://www.youtube.com/api/timedtext?v={VIDEO_ID}&lang=en&fmt=vtt")
        if "-->" in body:
            return body, "YouTube 자막"
        print("유튜브 직접 자막 조회: 빈 응답 → MIT 공식 WebVTT 사용")
    except Exception as e:
        print(f"유튜브 직접 자막 조회 실패({type(e).__name__}) → MIT 공식 WebVTT 사용")
    return fetch(MIT_VTT), MIT_VTT

def load_video_docs(minutes=15, window=30):
    """앞 minutes분 자막을 window초 묶음으로 → Document(본문=자막, metadata=시작 시각·재생 링크·자막 출처)"""
    try:
        vtt, caption_source = caption_text()
    except Exception as e:
        print(f"자막을 가져오지 못했습니다({type(e).__name__}). 영상 원문: {VIDEO_PAGE}")
        return []
    groups = {}
    for start, text in re.findall(r"(\d\d:\d\d:\d\d\.\d{3}) --> [^\n]*\n(.*?)(?:\n\n|\Z)", vtt, flags=re.S):
        sec = seconds(start)
        text = " ".join(text.split())
        if text and sec < minutes * 60:
            groups.setdefault(int(sec // window) * window, []).append(text)
    return [Document(page_content=" ".join(lines), metadata={
                "media": "video", "doc_id": f"video-{t:04d}s", "start_sec": t, "time": f"{t // 60:02d}:{t % 60:02d}",
                "url": f"{VIDEO_PAGE}&t={t}s", "thumbnail": f"https://i.ytimg.com/vi/{VIDEO_ID}/hqdefault.jpg",
                "title": VIDEO_TITLE, "caption_source": caption_source})
            for t, lines in sorted(groups.items())]

def load_figure_docs():
    """기사 본문 그림의 실제 캡션(▲ …)과 대체 텍스트 → Document(본문=캡션·대체 텍스트, metadata=이미지 URL·기사 출처)"""
    try:
        page = fetch(ARTICLE_URL)
    except Exception as e:
        print(f"기사를 가져오지 못했습니다({type(e).__name__}). 기사 원문: {ARTICLE_URL}")
        return []
    docs, seen = [], set()
    for m in re.finditer(r'<img[^>]+src="([^"]+etching-a-circuit-pattern_\d+_pc[^"]*)"[^>]*alt="([^"]*)"', page):
        src, alt = html.unescape(m.group(1)), html.unescape(m.group(2)).strip()
        after = re.sub(r"<[^>]+>", " ", page[m.end():m.end() + 1500])
        cap = re.search(r"▲\s*([^▲<]{2,60}?)\s{2,}", html.unescape(after))
        if src in seen or not cap:            # 같은 그림의 반응형 중복과 캡션 없는 머리 그림은 건너뜀
            continue
        seen.add(src)
        caption = "▲ " + cap.group(1).strip()
        docs.append(Document(page_content=f"그림 캡션: {caption} / 대체 텍스트: {alt}", metadata={
            "media": "figure", "doc_id": f"figure-{len(docs) + 1}", "caption": caption, "alt": alt,
            "image_url": src, "url": ARTICLE_URL, "title": ARTICLE_TITLE}))
    return docs

def open_media_index(docs):
    key = EMBEDDING_MODEL_NAME + "".join(d.page_content for d in docs)
    name = "media-" + hashlib.sha256(key.encode("utf-8")).hexdigest()[:8]
    media_store = Chroma(collection_name=name, embedding_function=embeddings, persist_directory="chroma_db",
                         collection_metadata={"hnsw:space": "cosine"})
    ids = [d.metadata["doc_id"] for d in docs]
    have = set(media_store.get(ids=ids)["ids"]) if ids else set()
    new = [d for d in docs if d.metadata["doc_id"] not in have]
    if new:
        media_store.add_documents(new, ids=[d.metadata["doc_id"] for d in new])
    print(f"영상·그림 컬렉션 {name} · 새로 추가 {len(new)}개 · 저장된 문서 {len(media_store.get()['ids'])}개")
    return media_store

def media_search(question, media, k=2):
    """media="video" 또는 "figure"로 메타데이터 필터를 걸고 검색한다."""
    if not MEDIA_DOCS:
        print("영상·그림 자료를 가져오지 못해 이 셀은 건너뜁니다. 위 셀의 원문 링크를 확인하세요.")
        return []
    found = media_store.similarity_search_with_score(question, k=k, filter={"media": media})
    return [{"rank": i, "distance": round(float(d), 4), "text": doc.page_content, **doc.metadata}
            for i, (doc, d) in enumerate(found, start=1)]

def show_media(hits):
    """검색 결과의 metadata에서 재생 링크·그림 URL을 꺼내 화면에 표시한다."""
    from IPython.display import HTML, display
    for h in hits:
        if h["media"] == "video":
            display(HTML(f'<div style="display:flex;gap:10px;align-items:flex-start;margin:6px 0">'
                         f'<img src="{h["thumbnail"]}" width="160" alt="영상 썸네일(식별용)">'
                         f'<div><b>{h["rank"]}위 · {h["time"]}부터</b> (거리 {h["distance"]})<br>{html.escape(h["text"])}<br>'
                         f'<a href="{h["url"]}" target="_blank">▶ {h["time"]}부터 재생</a> · 자막 출처: {html.escape(h["caption_source"])}</div></div>'))
        else:
            display(HTML(f'<div style="margin:6px 0"><b>{h["rank"]}위 · {html.escape(h["caption"])}</b> (거리 {h["distance"]})<br>'
                         f'<img src="{h["image_url"]}" width="420" alt="{html.escape(h["alt"])}"><br>'
                         f'대체 텍스트: {html.escape(h["alt"])} · <a href="{h["url"]}" target="_blank">기사 원문</a></div>'))

MEDIA_DOCS = load_video_docs() + load_figure_docs()
media_store = open_media_index(MEDIA_DOCS) if MEDIA_DOCS else None

def media_table(question, media, k=2):
    hits = media_search(question, media, k)
    return pd.DataFrame([{"질문": question, "순위": h["rank"], "문서": h["doc_id"], "거리": h["distance"],
                          "돌아갈 곳": h["url"], "검색된 글자": h["text"][:70]} for h in hits])

In [ ]:
display(pd.concat([media_table("습식 식각을 언급하는 영상 구간은?", "video"), media_table("wet etching을 설명하기 시작하는 시각은?", "video")]))
display(pd.concat([media_table("플라즈마 생성 과정을 보여주는 그림은?", "figure"), media_table("기체가 이온화되는 원리를 그린 그림은?", "figure")]))

**여기까지가 필수입니다.** 이제 VS Code에서 `바이브코딩_문서RAG` 실습으로 넘어갑니다.

---
## 선택 A. BM25 · 벡터 · MMR 나란히 보기

- **BM25**: 질문과 조각에 같은 글자 2개 묶음(한국어는 띄어쓰기만으로 자르기 어려워 글자 2-gram 사용)이 얼마나 겹치는지로 점수를 매깁니다. 임베딩을 쓰지 않습니다.
- **벡터**: 위에서 쓴 Chroma 코사인 거리 검색입니다.
- **MMR**: 벡터 후보 `fetch_k=5`개 중에서 질문 관련성과 이미 고른 조각과의 중복을 함께 따져 3개를 고릅니다(`lambda_mult=0.5`).
- 세 방식의 점수 체계는 서로 다르므로 숫자를 비교하지 않고 **반환된 조각 ID와 순위**만 비교합니다. 이 쪽은 조각이 5개뿐이라 후보 수(fetch_k)는 5가 최대입니다.

In [ ]:
import math
from collections import Counter

def grams(text):
    t = "".join(text.split())
    return [t[i:i + 2] for i in range(len(t) - 1)]

def bm25(question, chunks, k=3, k1=1.5, b=0.75):
    docs = [grams(c.page_content) for c in chunks]
    avg = sum(map(len, docs)) / len(docs)
    df = Counter(g for d in docs for g in set(d))
    def score(d):
        tf = Counter(d)
        return sum(math.log(1 + (len(docs) - df[g] + 0.5) / (df[g] + 0.5)) * tf[g] * (k1 + 1)
                   / (tf[g] + k1 * (1 - b + b * len(d) / avg)) for g in set(grams(question)) if g in tf)
    ranked = sorted(zip(chunks, map(score, docs)), key=lambda x: -x[1])[:k]
    return [{"rank": i, "distance": round(s, 3), "text": c.page_content, **c.metadata} for i, (c, s) in enumerate(ranked, start=1)]

mmr = [{"rank": i, "distance": "—", "text": d.page_content, **d.metadata}
       for i, d in enumerate(store.max_marginal_relevance_search(base_q, k=3, fetch_k=5, lambda_mult=0.5), start=1)]
print("벡터(기준)"); display(table(base_hits))
print("BM25(점수 열은 BM25 점수, 클수록 가까움)"); display(table(bm25(base_q, CHUNKS), base=base_hits))
print("MMR"); display(table(mmr, base=base_hits))

## 선택 B. 메타데이터 필터

같은 질문으로 영상 자막 중 **처음 5분(300초 미만)** 만 검색합니다. 필터와 일치하는 자료만 후보가 됩니다.

In [ ]:
if MEDIA_DOCS:
    found = media_store.similarity_search_with_score("습식 식각을 언급하는 영상 구간은?", k=2, filter={"$and": [{"media": "video"}, {"start_sec": {"$lt": 300}}]})
    for doc, d in found:
        print(doc.metadata["time"], round(float(d), 4), doc.metadata["url"], "|", doc.page_content[:80])

## 선택 C. 질문 임베딩 3D

문서 조각 벡터로 PCA 좌표계를 한 번 맞추고, 같은 모델로 임베딩한 질문을 그 좌표계에 **투영**합니다. 강조된 점은 Chroma가 실제로 반환한 조각(순위 번호)입니다. 순위는 원래 벡터 공간에서 정해진 것이며 3D 거리로 다시 정렬하지 않습니다. 조각이 5개뿐이라 3축이 설명하는 비율을 함께 출력합니다.

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import plotly.graph_objects as go

data = store.get(include=["embeddings", "documents"])
X = np.array(data["embeddings"])
pca = PCA(n_components=min(3, len(X) - 1)).fit(X)
P = pca.transform(X)
P = np.hstack([P, np.zeros((len(P), 3 - P.shape[1]))])
print(f"PCA {pca.n_components_}축이 설명하는 분산 비율 합계: {pca.explained_variance_ratio_.sum():.1%}")
fig = go.Figure(go.Scatter3d(x=P[:, 0], y=P[:, 1], z=P[:, 2], mode="markers+text", text=data["ids"], name="조각",
                             marker=dict(size=5, color="#9aa3b5"), hovertext=[t[:60] for t in data["documents"]], hoverinfo="text"))
for q, color in [(base_q, "#d62728"), ("식각 공정이란?", "#2ca02c")]:
    Q = pca.transform(np.array(embeddings.embed_query(q)).reshape(1, -1))
    Q = np.hstack([Q, np.zeros((1, 3 - Q.shape[1]))])
    hits = search(store, q, k=3)
    idx = [data["ids"].index(h["chunk_id"]) for h in hits]
    fig.add_trace(go.Scatter3d(x=[Q[0, 0]], y=[Q[0, 1]], z=[Q[0, 2]], mode="markers", name=f"질문: {q[:14]}…",
                               marker=dict(size=9, symbol="diamond", color=color)))
    fig.add_trace(go.Scatter3d(x=P[idx, 0], y=P[idx, 1], z=P[idx, 2], mode="markers+text", text=[f"{h['rank']}위" for h in hits],
                               name=f"반환 조각({q[:8]}…)", marker=dict(size=8, color="rgba(0,0,0,0)", line=dict(color=color, width=3))))
fig.update_layout(height=520, margin=dict(l=0, r=0, t=30, b=0))
fig.show()